# Session 6: dates and shapes, ready for charts

Session 7 is all charts. A chart needs two things from you before it can
be drawn, and this notebook makes both:

1. **dates that are real dates**, so you can ask for the month, the year or
   the day of the week, and add up per month
2. **a table in the right shape**: one row per point in time or per
   category, one column per line or per group of bars

Short notebook, three new tools: `.dt`, `resample`, and `pivot_table` with
its little sibling `unstack`.

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

here = Path.cwd()
while not (here / "data" / "clean" / "plays.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

plays = pd.read_csv("data/clean/plays.csv")
print(plays.shape)
print(plays["played_at"].dtype)

## 1. Text that looks like a date

`played_at` is `object`: text. So far that was fine, because ISO dates like
`"2025-01-01"` sort and compare correctly as text, and `.str[:4]` cuts out
the year. But text cannot tell you which day of the week it was, or which
month comes after December.

Convert it, naming the format on purpose, the lesson of notebook 04:

In [ ]:
plays["played_at"] = pd.to_datetime(plays["played_at"], format="%Y-%m-%d")
print(plays["played_at"].dtype)
print(plays["played_at"].min(), "to", plays["played_at"].max())

`datetime64[ns]` is pandas for "a real date and time". The data runs from
the 1st of January 2024 to the 31st of December 2025.

A shortcut you will see a lot: `pd.read_csv(path, parse_dates=["played_at"])`
converts while loading. Fine for a clean ISO file like this one. For a file
like the messy one, convert yourself, format by format.

## 2. What a real date can do: `.dt`

Text columns have `.str`. Date columns have **`.dt`**, which asks the same
question of every date at once.

In [ ]:
dates = plays["played_at"]

pd.DataFrame({
    "date": dates,
    "year": dates.dt.year,
    "month": dates.dt.month,
    "weekday": dates.dt.day_name(),
    "month as a period": dates.dt.to_period("M"),
}).head()

Plays per year, which SQL needed `strftime` or `substr` for, without
slicing any text:

In [ ]:
per_year = dates.dt.year.value_counts().sort_index()

assert per_year.to_dict() == {2024: 1122, 2025: 1061}
per_year

And a question text could not answer at all: which day of the week?

In [ ]:
days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
per_weekday = dates.dt.day_name().value_counts().reindex(days)

assert (per_weekday.min(), per_weekday.max()) == (302, 333)
per_weekday

Between 302 and 333 plays on every day: no favourite day. `.reindex(days)`
puts the days in calendar order instead of biggest first, which is the
order a chart of weekdays wants.

## 3. Per month, two ways

**With `groupby`**: group by the month each date falls in.
`to_period("M")` turns 2024-10-17 into `2024-10`.

In [ ]:
month = dates.dt.to_period("M")
plays_per_month = plays.groupby(month).size()

assert len(plays_per_month) == 24
assert plays_per_month.idxmax() == pd.Period("2024-10", "M") and plays_per_month.max() == 127
assert plays_per_month.idxmin() == pd.Period("2025-08", "M") and plays_per_month.min() == 57
plays_per_month

24 months. The busiest was October 2024 with 127 plays, the quietest August
2025 with 57. That is a **time series**: one number per point in time, in
order. It is what a line chart draws.

**With `resample`**: made for exactly this. It needs the dates as the
index, the row labels, and a rule: `"MS"` means "month, labelled by its
start".

In [ ]:
minutes_per_month = plays.set_index("played_at")["minutes_played"].resample("MS").sum()

assert len(minutes_per_month) == 24
assert round(minutes_per_month.max(), 1) == 554.6          # October 2024
assert round(minutes_per_month.min(), 1) == 231.1          # June 2025
minutes_per_month.round(1)

Most minutes in October 2024, 554.6; fewest in June 2025, 231.1. And in
both years the three quietest months are June, July and August:

In [ ]:
quietest = minutes_per_month.groupby(minutes_per_month.index.year).nsmallest(3)
assert sorted(set(quietest.index.get_level_values(1).month)) == [6, 7, 8]
quietest.round(1)

The difference between the two: `groupby` only makes a row for a month
that has data. `resample` makes a row for **every** month in the range,
even an empty one. Usually that is what you want for a chart, because a
missing month should look missing, not skipped. But it brings one trap.

### The sum of nothing is zero

Remember May 2024 on the tablet from notebook 04: one play, and its minutes
were blank. Take the cleaned messy file and add up the tablet minutes per
month.

In [ ]:
sys.path.insert(0, "sessions/session-06/solutions")
from cleaning import clean_plays, load_messy

cleaned = clean_plays(load_messy("data/messy/plays_messy.csv"))
cleaned["played_at"] = pd.to_datetime(cleaned["played_at"], format="%Y-%m-%d")

tablet = cleaned[cleaned["device"] == "tablet"].set_index("played_at")["minutes_played"]

pd.DataFrame({
    "sum": tablet.resample("MS").sum(),
    "mean": tablet.resample("MS").mean(),
    "sum, min_count=1": tablet.resample("MS").sum(min_count=1),
}).head(6)

In [ ]:
may = pd.Timestamp("2024-05-01")
assert tablet.resample("MS").sum()[may] == 0
assert pd.isna(tablet.resample("MS").mean()[may])
assert pd.isna(tablet.resample("MS").sum(min_count=1)[may])

The mean of nothing is `NaN`, as notebook 04 found. But the **sum** of
nothing is `0.0`, and that is a claim: "you listened to the tablet for zero
minutes in May". It is `fillna(0)` again, done for you without asking. A line
chart of it would dip to the floor in May.

`min_count=1` says "only add up if there is at least one number", and gives
`NaN` back for May. Not knowing stays not knowing.

## 4. Long and wide

Plays per month **per device**. `groupby` with two columns gives one row
per pair:

In [ ]:
long = plays.groupby([month, "device"]).size()

assert len(long) == 24 * 5
long.head(7)

That is **long** format: 120 rows, one per month and device. Good for
storing, awkward to read, and not what a chart wants. A chart with one line
per device wants **wide** format: one row per month, one column per device.

`unstack()` takes the inner level of the index, the device, and turns it
into columns:

In [ ]:
wide = long.unstack()

assert wide.shape == (24, 5)
wide.head()

Same 120 numbers, now 24 rows by 5 columns. `pivot_table` builds the same
table in one call, and says what it is doing in its arguments: rows,
columns, which values, and how to combine them.

In [ ]:
pivot = plays.pivot_table(index=month, columns="device", values="play_id", aggfunc="count")

assert pivot.equals(wide)
pivot.head()

Swap `aggfunc="count"` for `"sum"` and `values` for `"minutes_played"`,
and it adds up minutes instead:

In [ ]:
minutes_by_device = plays.pivot_table(index=month, columns="device",
                                      values="minutes_played", aggfunc="sum")

assert round(minutes_by_device["phone"].sum(), 1) == 4308.2
minutes_by_device.round(1).head()

### The session 4 stretch, in one line

In the session 4 stretch you built plays per genre per year by hand, with a
dictionary of dictionaries. Here it is with `pivot_table`, plus the change
column:

In [ ]:
plays["year"] = dates.dt.year
genre_by_year = plays.pivot_table(index="genre", columns="year", values="play_id", aggfunc="count")
genre_by_year["change"] = genre_by_year[2025] - genre_by_year[2024]

assert genre_by_year.loc["Folk"].tolist() == [236, 202, -34]
assert genre_by_year["change"].idxmin() == "Folk"
assert genre_by_year[[2024, 2025]].sum().sum() == len(plays)
genre_by_year.sort_values("change")

Folk fell most, by 34 plays; only Jazz and Afrobeat grew. The last `assert`
is the check from the session 4 stretch: the cells add up to every play.

### When a gap in a wide table really is zero

Plays per month per **artist** has gaps: not every artist was played every
month.

In [ ]:
by_artist = plays.groupby([month, "artist_name"]).size().unstack()
print(by_artist.shape, "with", by_artist.isna().sum().sum(), "empty cells")

by_artist = plays.groupby([month, "artist_name"]).size().unstack(fill_value=0)
assert by_artist.isna().sum().sum() == 0
assert by_artist.sum().sum() == len(plays)

275 empty cells. Here `fill_value=0` is the truth, not a claim: a month in
which an artist has no rows is a month with zero plays of that artist. Compare
the minutes in notebook 04, where a blank meant "nobody wrote it down", and
0 would have been a lie. The question is always the same: **does the gap mean
none, or unknown?**

## 5. A glance, not a chart

Every DataFrame has a `.plot()`. On a wide table it draws one line per
column, which is a quick way to see whether the shape is right:

In [ ]:
wide.plot()

No title, no answer, colours chosen by nobody. It is a glance, and that is
all it is for. Session 7 turns these tables into charts that answer a
question:

| Table | Shape | The chart it becomes |
|---|---|---|
| `plays_per_month`, `minutes_per_month` | a time series, 24 rows | a line |
| `wide`, `minutes_by_device` | wide, 24 months by 5 devices | several lines, or stacked bars |
| `genre_by_year` | wide, 8 genres by 2 years | grouped bars |

## What you have seen

| Idea | pandas |
|---|---|
| text to dates, on purpose | `pd.to_datetime(col, format="%Y-%m-%d")` |
| parts of a date | `.dt.year`, `.dt.month`, `.dt.day_name()` |
| the month a date falls in | `.dt.to_period("M")` |
| a total per month, every month | `.set_index("played_at").resample("MS").sum()` |
| a sum of nothing is not zero | `.sum(min_count=1)` |
| long to wide | `.unstack()`, or `.pivot_table(index=, columns=, values=, aggfunc=)` |
| a gap that really is zero | `fill_value=0` |

## Next

`06-cleaning-functions-exercise.ipynb`: the cleaning from notebook 04, kept
as two functions.